# YOLOv12n baseline

Trains the stock YOLOv12n detector on PKU-Market-PCB (100 epochs, `imgsz=640`).

On Colab the dataset is cached onto the VM SSD and runs are synced back to Drive. Locally, `project_paths.py` uses `PCB_DATA_YOLO/` and `runs_pcb/` in this repo. An existing `best.pt` is reused so a re-run does not overwrite the published checkpoint.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.chdir('/content/drive/MyDrive/lld-net-pcb-ml')
except ImportError:
    pass

from project_paths import setup, prepare_dataset, sync_run

P = setup()
os.chdir(P.repo_root)
print('repo :', P.repo_root)
print('colab:', P.in_colab)

In [ ]:
import shutil

if shutil.which('nvidia-smi'):
    !nvidia-smi
else:
    print('nvidia-smi not found — CPU / no NVIDIA driver.')

try:
    import google.colab
    %pip -q install ultralytics pyyaml
except ImportError:
    pass

In [ ]:
import os
import shutil
from pathlib import Path
import yaml
import ultralytics
from ultralytics import YOLO

print('Ultralytics:', ultralytics.__version__)

In [ ]:
REPO_ROOT          = P.repo_root
DRIVE_DATASET_DIR  = P.drive_dataset
DRIVE_DATA_YAML    = P.drive_data_yaml
DRIVE_RUNS_DIR     = P.drive_runs

LOCAL_DATASET_DIR  = P.local_dataset
LOCAL_DATA_YAML    = P.local_data_yaml
LOCAL_PROJECT_DIR  = P.local_runs

DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('REPO_ROOT      :', REPO_ROOT)
print('LOCAL_DATASET  :', LOCAL_DATASET_DIR)

In [ ]:
# Colab: cache Drive → VM SSD. Local: use PCB_DATA_YOLO/ in place.
prepare_dataset(P)
print('Local cache:', LOCAL_DATASET_DIR)

In [ ]:
for split in ['train', 'val', 'test']:
    img_dir = LOCAL_DATASET_DIR / 'images' / split
    lbl_dir = LOCAL_DATASET_DIR / 'labels' / split
    n_img = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f'{split:5s} -> images: {n_img:5d}, labels: {n_lbl:5d}')

In [ ]:
MODEL_WEIGHTS = 'yolo12n.pt'   # auto-downloaded by Ultralytics on first use
RUN_NAME      = 'yolo12_pcb_baseline'

IMG_SIZE     = 640
EPOCHS       = 100
BATCH        = 16
DEVICE       = 0 if __import__('torch').cuda.is_available() else 'cpu'
WORKERS      = 4
PATIENCE     = 30
SAVE_PERIOD  = 5
SEED         = 42

In [ ]:
LOCAL_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
run_dir   = LOCAL_PROJECT_DIR / RUN_NAME
last_ckpt = run_dir / 'weights' / 'last.pt'
best_pt   = run_dir / 'weights' / 'best.pt'

# If a previous run already lives on Drive, copy it back to local SSD so resume can pick up.
drive_run_dir = DRIVE_RUNS_DIR / RUN_NAME
if drive_run_dir.exists() and not run_dir.exists():
    shutil.copytree(drive_run_dir, run_dir)

if last_ckpt.exists():
    print('Resuming from:', last_ckpt)
    model = YOLO(str(last_ckpt))
    train_results = model.train(resume=True)
elif best_pt.exists():
    print('Existing checkpoint found — skip training:', best_pt)
    print('Delete best.pt if you want to retrain from scratch.')
    model = YOLO(str(best_pt))
else:
    model = YOLO(MODEL_WEIGHTS)
    train_results = model.train(
        data        = str(LOCAL_DATA_YAML),
        epochs      = EPOCHS,
        imgsz       = IMG_SIZE,
        batch       = BATCH,
        device      = DEVICE,
        workers     = WORKERS,
        project     = str(LOCAL_PROJECT_DIR),
        name        = RUN_NAME,
        exist_ok    = True,
        pretrained  = True,
        cache       = True,
        verbose     = True,
        patience    = PATIENCE,
        save_period = SAVE_PERIOD,
        seed        = SEED,
        deterministic = True,
    )

if not best_pt.exists():
    raise FileNotFoundError(f'best.pt not found: {best_pt}')
print('best.pt:', best_pt)

In [ ]:
# Colab: copy the run back to Drive. Local: no-op (same folder).
drive_run_dir = DRIVE_RUNS_DIR / RUN_NAME
sync_run(run_dir, drive_run_dir)

In [ ]:
best_model = YOLO(str(best_pt))
val_metrics  = best_model.val(data=str(LOCAL_DATA_YAML), split='val',  imgsz=IMG_SIZE, device=DEVICE)
test_metrics = best_model.val(data=str(LOCAL_DATA_YAML), split='test', imgsz=IMG_SIZE, device=DEVICE)
print('VAL  :', val_metrics.results_dict)
print('TEST :', test_metrics.results_dict)

In [ ]:
# Small visual gallery — not the full 2k-image test split.
PRED_NAME = f'{RUN_NAME}_predict'
test_dir = LOCAL_DATASET_DIR / 'images' / 'test'
samples = sorted(
    p for p in test_dir.iterdir()
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}
)
step = max(1, len(samples) // 16)
gallery = samples[::step][:16]

best_model.predict(
    source  = [str(p) for p in gallery],
    imgsz   = IMG_SIZE,
    conf    = 0.25,
    save    = True,
    project = str(LOCAL_PROJECT_DIR),
    name    = PRED_NAME,
    exist_ok= True,
    verbose = False,
)
sync_run(LOCAL_PROJECT_DIR / PRED_NAME, DRIVE_RUNS_DIR / PRED_NAME)

In [ ]:
# Optional: ONNX export for deployment.
onnx_path = best_model.export(format='onnx', imgsz=IMG_SIZE)
onnx_p = Path(onnx_path)
if onnx_p.exists():
    drive_onnx = DRIVE_RUNS_DIR / RUN_NAME / 'weights' / onnx_p.name
    drive_onnx.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(onnx_p, drive_onnx)
    print('ONNX:', drive_onnx)